# Web Scraping

# Tarefa 1 - Wikipedia (Steve Jobs)

In [8]:
import requests
from bs4 import BeautifulSoup
import os
import time
import csv
from datetime import datetime

In [9]:
BASE_URL    = "https://pt.wikipedia.org"
url_inicial = "https://pt.wikipedia.org/wiki/Steve_Jobs"

response = requests.get(url_inicial) #HTML da pg inicial
print(response)

os.makedirs("html_pages", exist_ok=True)

with open("html_pages/Steve_Jobs.html", "w", encoding="utf-8") as f:
    f.write(response.text)

<Response [403]>


In [10]:
soup = BeautifulSoup(response.text, "html.parser")

ignorar = ("Categoria:", "Ficheiro:", "Especial:", "Wikipedia:", "Ajuda:", "Portal:", "Predefinição:")

links_internos = [] #extrai links internos /wiki
for tag in soup.find_all("a", href=True):
    href = tag.get("href")
    if href.startswith("/wiki/") and "#" not in href:
        if not any(p in href for p in ignorar):
            url_completa = BASE_URL + href
            if url_completa not in links_internos:
                links_internos.append(url_completa)

print(f"{len(links_internos)} links internos encontrados")

0 links internos encontrados


In [11]:
paginas = {url_inicial: response.text} #baixa o HTML da pg  inicial

for i, url in enumerate(links_internos):
    try:
        resp = requests.get(url, timeout=10)
        paginas[url] = resp.text

        nome = url.split("/wiki/")[-1][:80] + ".html"
        with open(f"html_pages/{nome}", "w", encoding="utf-8") as f:
            f.write(resp.text)

    except Exception as e:
        print(f"Erro em {url}: {e}")

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(links_internos)} páginas baixadas...")

    time.sleep(0.5) #delay de 5s

print(f"Total de páginas baixadas: {len(paginas)}")

Total de páginas baixadas: 1


In [ ]:
dados_paginas = []
dados_imagens = []

for url, html in paginas.items():
    s = BeautifulSoup(html, "html.parser")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    tag_titulo = s.find(id="firstHeading") #título da pg
    titulo = tag_titulo.get_text(strip=True) if tag_titulo else "N/A"

    conteudo = s.find(id="mw-content-text") #primeiro paragrafo do artigo
    primeiro_paragrafo = "N/A"
    if conteudo:
        for p in conteudo.find_all("p"):
            texto = p.get_text(strip=True)
            if len(texto) > 30: #mais de 30 caracteres
                primeiro_paragrafo = texto
                break